In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("dynamicPartitionPruningApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.optimizer.dynamicPartitionPruning.enabled", "false")
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/25 21:41:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
yellowTaxiSchema = StructType([
 StructField("VendorID", IntegerType(), True),
 StructField("tpep_pickup_datetime", TimestampType(), True),
 StructField("tpep_dropoff_datetime", TimestampType(), True),
 StructField("passenger_count", DoubleType(), True),
 StructField("trip_distance", DoubleType(), True),
 StructField("RatecodeID", DoubleType(), True),
 StructField("store_and_fwd_flag", StringType(), True),
 StructField("PULocationID", IntegerType(), True),
 StructField("DOLocationID", IntegerType(), True),
 StructField("payment_type", IntegerType(), True),
 StructField("fare_amount", DoubleType(), True),
 StructField("extra", DoubleType(), True),
 StructField("mta_tax", DoubleType(), True),
 StructField("tip_amount", DoubleType(), True),
 StructField("tolls_amount", DoubleType(), True),
 StructField("improvement_surcharge", DoubleType(), True),
 StructField("total_amount", DoubleType(), True),
 StructField("congestion_surcharge", DoubleType(), True),
 StructField("airport_fee", DoubleType(), True),
])

yellowTaxisDF = spark.read.option("header", "true").schema(yellowTaxiSchema).csv(
    "./Files/YellowTaxis_202210.csv"
)

# Save yellowTaxis as a partitioned table
yellowTaxisDF.write.partitionBy("PULocationID").option("header", "true").option("dateFormat", "yyyy-MM-dd HH:mm:ss.S").mode("overwrite").format("csv").option("path","./Files/Output/YellowTaxisPartitionedOutputCSV.csv").saveAsTable("YellowTaxis")

In [8]:
taxiZonesSchema = 'LocationID INT, Borough STRING, Zone STRING, ServiceZone STRING'

taxiZonesDF = spark.read.schema(taxiZonesSchema).csv(
    "./Files/TaxiZones.csv"
)

taxiZonesDF.write.option("header", "true").option("dateFormat", "yyyy-MM-dd HH:mm:ss.S").mode("overwrite").format("csv").option("path","./Files/Output/TaxiZones.csv").saveAsTable("TaxiZones")

In [9]:
# Join YellowTaxis with TaxiZones using dynamic partition pruning
spark.sql("""
SELECT * FROM
    YellowTaxis y
    JOIN
    TaxiZones z
    ON y.PULocationID = z.LocationID
    WHERE y.PULocationID = 1
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+----------+-------+--------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|PULocationID|LocationID|Borough|          Zone|ServiceZone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+----------+-------+--------------+-----------+
|       2| 2022-10-09 18:42:58|  2022-10-09 18:43

In [10]:
spark.sql("""
SELECT * FROM TaxiZones WHERE Borough = 'EWR'
""").show()

+----------+-------+--------------+-----------+
|LocationID|Borough|          Zone|ServiceZone|
+----------+-------+--------------+-----------+
|         1|    EWR|Newark Airport|        EWR|
+----------+-------+--------------+-----------+



In [11]:
spark.sql("""
SELECT * FROM
    YellowTaxis y
    JOIN
    TaxiZones z
    ON y.PULocationID = z.LocationID
    WHERE z.Borough = 'EWR'
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+----------+-------+--------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|PULocationID|LocationID|Borough|          Zone|ServiceZone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+----------+-------+--------------+-----------+
|       2| 2022-10-09 18:42:58|  2022-10-09 18:43

In [12]:
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")

In [ ]:
spark.sql("""
SELECT * FROM
    YellowTaxis y
    JOIN
    TaxiZones z
    ON y.PULocationID = z.LocationID
    WHERE z.Borough = 'EWR'
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+----------+-------+--------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|PULocationID|LocationID|Borough|          Zone|ServiceZone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+----------+-------+--------------+-----------+
|       2| 2022-10-09 18:42:58|  2022-10-09 18:43

25/06/26 03:25:12 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 938074 ms exceeds timeout 120000 ms
25/06/26 03:25:12 WARN SparkContext: Killing executors is not supported by current scheduler.
25/06/26 03:25:13 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$